# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook guides you through loading and exploring the mental health survey dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View metadata: access as a single object
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Dataset description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"@id: {metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities are referenced by their `@id` fields.

In [ ]:
# List all record sets available in the dataset
record_sets = list(dataset.record_sets)
print(f"Record Sets Found ({len(record_sets)}):")
for rs in record_sets:
    print(f"- @id: {rs.id}, Name: {rs.name}, Description: {getattr(rs, 'description', None)}")

# For each record set, list associated fields and columns (@id references)
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    fields = rs.fields
    print("  Fields:")
    for f in fields:
        # field may have: id, name, data_type, description
        print(f"    - {f.name} (@id: {f.id}, data_type: {getattr(f, 'data_type', None)})")
    columns = getattr(rs, 'columns', [])
    if columns:
        print("  Columns:")
        for c in columns:
            print(f"    - {c.name} (@id: {c.id}, data_type: {getattr(c, 'data_type', None)})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll use the first record set and its fields as identified by their `@id` values.

In [ ]:
# Choose a record set and extract records
dataframes = {}

# Use @id to reference record sets
record_sets_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_sets_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    # Fetch using mlcroissant records generator
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"No records found for {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Fields/columns in '{record_set_id}':", df.columns.tolist())
    print(df.head(3))

# Select the first record set with loaded records for further analysis
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"\nSelected record set for EDA: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping, and handling outliers.

You'll need to reference fields by their `@id` (column names for Pandas DataFrame), such as GAD-7 score, PHQ-9 score, or demographic data.

In [ ]:
# Choose a numeric field for processing
# Let's inspect and select a numeric field (@id) from the DataFrame
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields available:", numeric_fields)

# Example: Choose 'gad7_score' or equivalent (@id, actual dataframe column name)
if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    print(filtered_df.head(3))

    # Normalize
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, normalized_col]].head())

    # Group by a demographic field, e.g., 'gender' or 'village', referencing by column @id
    group_field_candidates = [col for col in df.columns if (df[col].dtype == 'object' and not any(substr in col.lower() for substr in ['score', 'id']))]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll create a histogram of the selected numeric field and a bar plot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Bar plot of mean score by group field (if exists)
    if 'group_field' in locals():
        plt.figure(figsize=(8, 5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
This exploration demonstrates how to use the `mlcroissant` library to load, analyze, and visualize data from a FAIR² Croissant dataset package.

- Entities and fields are referenced by their `@id` for reproducibility and consistency.
- Data extraction and EDA reveal key features, distributions, and demographic patterns within the survey dataset.
- The dataset enables community-centric insights into mental health in Kilifi, Kenya.

For advanced analysis, you can further investigate correlations, feature importance, or predictive models using the extracted, AI-ready data.